# 04 · Walk network (pgRouting 4) and snapping

Same code as `make network` plus the origin/destination steps of `make score`.

- **Walkable:** every Overture road segment except `motorway`, and except segments whose access
  rules deny foot travel (no rule allowing it). `as_private` counts as walkable.
- **Topology:** segments are split at their Overture connectors (`ST_LineSubstring` between
  consecutive `at` values). Connector ids become bigint vertex ids. Undirected, cost = length (m).
  pgRouting 4 has no `pgr_createTopology`; none is needed.
- **Components:** `pgr_connectedComponents`; routing uses the largest component only.
- **Snapping:** origins (residential buildings, block-group centroids) and destinations are
  projected onto the nearest main edge. Distance = snap + along-edge + network. Destinations are
  virtual nodes attached to both ends of their edge.

In [ ]:
import os

import urban_access  # noqa: F401  (first: sets PROJ_DATA before geopandas/pyogrio load)
from urban_access.config import load_city

CITY = os.environ.get("CITY", "carson_city")
cfg = load_city(CITY)
print(cfg.name, "| Overture", cfg.overture_release, "| ACS", cfg.acs_year)
print("schema:", cfg.slug, "| EPSG", cfg.analysis_epsg)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from urban_access import db, network

eng = db.engine()


def q(sql: str, **params) -> pd.DataFrame:
    with eng.connect() as con:
        return pd.read_sql(text(sql.replace("__S__", f'"{cfg.slug}"')), con, params=params)


def qgeo(sql: str) -> gpd.GeoDataFrame:
    with eng.connect() as con:
        return gpd.read_postgis(text(sql.replace("__S__", f'"{cfg.slug}"')), con, geom_col="geom")

In [ ]:
stats = network.build(cfg)
stats.round(2)

In [ ]:
print(
    q("""SELECT class, count(*) segments,
                  count(*) FILTER (WHERE class = 'motorway') AS dropped_motorway,
                  count(*) FILTER (WHERE NOT foot_ok) AS dropped_foot_denied
           FROM __S__.walk_segments GROUP BY 1 ORDER BY 2 DESC""").to_string()
)
comp = q("""SELECT component, count(*) vertices FROM __S__.vertices
              GROUP BY 1 ORDER BY 2 DESC""")
print(
    f"{len(comp)} components; largest {comp['vertices'].iloc[0]:,} vertices "
    f"({100 * comp['vertices'].iloc[0] / comp['vertices'].sum():.1f} %); next sizes:",
    comp["vertices"].iloc[1:8].tolist(),
)
assert stats["main_length_pct"] > 95

In [ ]:
edges = qgeo("SELECT id, class, main, geom FROM __S__.edges")
dropped = qgeo("""SELECT class, foot_ok, geom FROM __S__.walk_segments
                   WHERE class = 'motorway' OR NOT foot_ok""")
bg = qgeo("SELECT geoid, geom FROM __S__.block_groups")
fig, ax = plt.subplots(figsize=(10, 12))
bg.boundary.plot(ax=ax, color="orange", linewidth=0.5)
edges[edges["main"]].plot(ax=ax, color="lightgray", linewidth=0.3)
edges[~edges["main"]].plot(ax=ax, color="blue", linewidth=1.5, label="not in main component")
dropped[dropped["class"] == "motorway"].plot(ax=ax, color="red", linewidth=1, label="motorway")
dropped[dropped["class"] != "motorway"].plot(
    ax=ax, color="purple", linewidth=1.5, label="foot denied"
)
ax.set_xlim(*bg.total_bounds[[0, 2]])
ax.set_ylim(*bg.total_bounds[[1, 3]])
ax.legend()
ax.set_title("Walk network: main component, excluded edges")
ax.set_axis_off()

## Origins, destinations, distances

In [ ]:
network.distances(cfg)  # sql/30_origins, 40_destinations, 50_distances

In [ ]:
snap = q("""SELECT 'origin' AS what, kind, snap_m FROM __S__.origins
              UNION ALL SELECT 'dest', service, snap_m FROM __S__.dests""")
print(snap.groupby(["what", "kind"])["snap_m"].describe(percentiles=[0.5, 0.95, 0.99]).round(1))
lim = cfg.scoring.max_snap_distance_m
flag = snap[snap["snap_m"] > lim]
print(f"snaps > {lim:g} m:", len(flag), f"({100 * len(flag) / len(snap):.2f} %)")
assert len(flag) / len(snap) < 0.01
fig, ax = plt.subplots(figsize=(8, 3))
snap[snap["what"] == "origin"]["snap_m"].clip(upper=300).hist(bins=60, ax=ax)
ax.axvline(lim, color="red")
ax.set_xlabel("origin snap distance (m, clipped at 300)")

In [ ]:
pop = q("""SELECT o.bg_geoid, sum(o.pop) AS pop_alloc, g.pop_total
             FROM __S__.origins o JOIN __S__.block_groups g ON g.geoid = o.bg_geoid
             GROUP BY o.bg_geoid, g.pop_total""")
assert ((pop["pop_alloc"] - pop["pop_total"]).abs() < 0.5).all(), "allocation != ACS"
print("population allocated:", round(pop["pop_alloc"].sum()), "=", int(pop["pop_total"].sum()))
print(
    q("""SELECT kind, count(*) n, round(sum(area_m2 * floors)::numeric) weight,
                  round(avg(floors)::numeric, 2) mean_floors
           FROM __S__.res_buildings GROUP BY 1""")
)

In [ ]:
# independent check: one-to-many pgr_dijkstraCost with the origin inserted as its own node
chk = network.check_pairs(cfg, n=10)
assert (chk["diff_m"].abs() < 0.01).all(), chk
chk.round(2)

In [ ]:
d = q("""SELECT od.service, od.dist_m, od.euclid_m FROM __S__.origin_dist od
           JOIN __S__.origins o ON o.id = od.origin_id WHERE o.kind = 'building'""")
assert not (d["dist_m"] < d["euclid_m"] - 0.01).any(), "network shorter than straight line"
d["circuity"] = d["dist_m"] / d["euclid_m"]
d.groupby("service").agg(
    reachable_pct=("dist_m", lambda s: 100 * s.notna().mean()),
    median_dist_m=("dist_m", "median"),
    median_circuity=("circuity", "median"),
).round(2)